# Analisis de alquileres scrapeados

Este notebook analiza el archivo generado por el scraper: `data/processed/infocasas_1_dormitorio_detalle.csv`.

El objetivo es revisar todos los tipos de alquiler disponibles en la muestra, no solo apartamentos. Para eso usamos `tipo_propiedad`, `barrio`, `moneda`, `monto`, `gastos_comunes` y `metros_cuadrados`.


## 1. Importar librerias y ubicar el proyecto

Esta celda busca la raiz del proyecto para que el notebook funcione aunque se abra desde `notebooks/` o desde la carpeta raiz.


In [ ]:
from pathlib import Path
import re

import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

def encontrar_raiz_proyecto() -> Path:
    actual = Path.cwd().resolve()
    for ruta in [actual, *actual.parents]:
        if (ruta / 'data' / 'processed' / 'infocasas_1_dormitorio_detalle.csv').exists():
            return ruta
    raise FileNotFoundError('No se encontro data/processed/infocasas_1_dormitorio_detalle.csv')

ROOT_DIR = encontrar_raiz_proyecto()
DATA_PATH = ROOT_DIR / 'data' / 'processed' / 'infocasas_1_dormitorio_detalle.csv'
ANALYSIS_DIR = ROOT_DIR / 'data' / 'processed' / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

ROOT_DIR


## 2. Cargar el dataset crudo


In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Filas: {len(df_raw)}')
print(f'Columnas: {len(df_raw.columns)}')
df_raw.head()


## 3. Limpieza basica

Normalizamos columnas numericas, convertimos gastos comunes a numero y calculamos costo mensual total. Tambien dejamos `precio_m2` como metrica derivada.


In [ ]:
def limpiar_monto_gastos(valor):
    if pd.isna(valor):
        return 0.0
    texto = str(valor).strip()
    if not texto:
        return 0.0
    numero = re.sub(r'[^0-9]', '', texto)
    return float(numero) if numero else 0.0

df = df_raw.copy()

for columna in ['monto', 'dormitorios', 'banios', 'metros_cuadrados']:
    df[columna] = pd.to_numeric(df[columna], errors='coerce')

df['gastos_comunes_monto'] = df['gastos_comunes'].apply(limpiar_monto_gastos)
df['costo_mensual_total'] = df['monto'] + df['gastos_comunes_monto']
df.loc[df['metros_cuadrados'] <= 0, 'metros_cuadrados'] = pd.NA
df['precio_m2'] = df['monto'] / df['metros_cuadrados']

df.to_csv(ANALYSIS_DIR / 'alquileres_limpios.csv', index=False)
df.head()


## 4. Calidad de datos


In [ ]:
faltantes = df.isna().sum().rename_axis('columna').reset_index(name='faltantes')
faltantes.to_csv(ANALYSIS_DIR / 'valores_faltantes.csv', index=False)
faltantes


## 5. Composicion de la muestra

Primero miramos que tipos de propiedad y barrios aparecen. Esto define que comparaciones tienen sentido.


In [ ]:
conteo_tipo = df['tipo_propiedad'].value_counts(dropna=False).rename_axis('tipo_propiedad').reset_index(name='cantidad')
conteo_barrio = df['barrio'].value_counts(dropna=False).rename_axis('barrio').reset_index(name='cantidad')

conteo_tipo.to_csv(ANALYSIS_DIR / 'conteo_tipo_propiedad.csv', index=False)
conteo_barrio.to_csv(ANALYSIS_DIR / 'conteo_barrio.csv', index=False)

display(conteo_tipo)
display(conteo_barrio.head(15))


## 6. Revision de moneda

No conviene comparar precios en pesos y dolares como si fueran la misma unidad. Primero analizamos por moneda.


In [ ]:
df['moneda'].value_counts(dropna=False)


## 7. Precio por tipo de propiedad


In [ ]:
precio_tipo = (
    df.groupby(['moneda', 'tipo_propiedad'], dropna=False)['monto']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .reset_index()
)
precio_tipo.to_csv(ANALYSIS_DIR / 'precio_por_tipo_propiedad.csv', index=False)
precio_tipo


## 8. Precio por barrio


In [ ]:
precio_barrio = (
    df.groupby(['moneda', 'barrio'], dropna=False)['monto']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .reset_index()
    .sort_values(['moneda', 'median'], na_position='last')
)
precio_barrio.to_csv(ANALYSIS_DIR / 'precio_por_barrio.csv', index=False)
precio_barrio


## 9. Costo mensual total


In [ ]:
costo_total_barrio = (
    df.groupby(['moneda', 'barrio'], dropna=False)['costo_mensual_total']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .reset_index()
    .sort_values(['moneda', 'median'], na_position='last')
)
costo_total_barrio.to_csv(ANALYSIS_DIR / 'costo_total_por_barrio.csv', index=False)
costo_total_barrio


## 10. Precio por metro cuadrado


In [ ]:
precio_m2_barrio = (
    df.dropna(subset=['precio_m2'])
    .groupby(['moneda', 'barrio'], dropna=False)['precio_m2']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .reset_index()
    .sort_values(['moneda', 'median'], na_position='last')
)
precio_m2_barrio.to_csv(ANALYSIS_DIR / 'precio_m2_por_barrio.csv', index=False)
precio_m2_barrio


## 11. Graficos

Estos graficos trabajan primero con publicaciones en pesos para no mezclar monedas.


In [ ]:
if plt is None:
    print('matplotlib no esta instalado. Instalar con: pip install matplotlib')
else:
    df_pesos = df[df['moneda'] == '$'].copy()
    df_pesos['monto'].dropna().hist()
    plt.title('Distribucion de alquileres en pesos')
    plt.xlabel('Monto')
    plt.ylabel('Cantidad')
    plt.show()


In [ ]:
if plt is not None:
    (
        df[df['moneda'] == '$']
        .groupby('barrio')['monto']
        .median()
        .sort_values()
        .plot(kind='barh')
    )
    plt.title('Precio mediano por barrio en pesos')
    plt.xlabel('Monto')
    plt.ylabel('Barrio')
    plt.show()


In [ ]:
if plt is not None:
    df['tipo_propiedad'].value_counts().plot(kind='bar')
    plt.title('Publicaciones por tipo de propiedad')
    plt.xlabel('Tipo de propiedad')
    plt.ylabel('Cantidad')
    plt.show()


## 12. Lectura inicial

Puntos a revisar antes de sacar conclusiones fuertes:

- La muestra actual es chica y viene de una busqueda especifica.
- Hay que separar precios por moneda o convertirlos a una moneda comun.
- `tipo_propiedad` permite comparar segmentos, pero algunos tipos pueden tener pocos casos.
- `referencia` sirve para deduplicar cuando se amplie el scraping.
- `metros_cuadrados` puede requerir revision manual si la publicacion informa varias superficies.
